# PDP-CR · Thí nghiệm theo kế hoạch của thầy Hùng

Chọn runtime **CPU**. Chạy ô 1, chọn `MODE` ở ô 2 rồi chạy ô 3. Ô 4 hiển thị bảng và tải các kết quả đã lưu.

| `MODE` | Thí nghiệm |
|---|---|
| `lr108` | Một instance của Thảo. TSR Thảo, TSR Đức, TSR Claude, GA thuần v2, GA memetic pilot và MLS. |
| `pilot` | Chín instance pilot gốc, ngân sách 30/60/120 giây CPU cho 50/100/200 tác vụ. So sánh TSR Đức, Claude, các GA và MLS. |
| `calibration` | Ba instance đối chứng của thầy. So sánh tốc độ TSR và thử các cấu hình restart, beta. |
| `families` | 15 instance gốc của Thảo, năm instance mỗi họ LC, LR và LRC. |
| `ablation` | 15 instance trên với các biến thể GA, MLS và TSR nhiều lần khởi động. |
| `claude_euclidean`, `claude_grid` | Campaign Claude nguyên gốc, chạy một lõi và dùng thời gian đồng hồ như specification. |
| `performance` | Đối chiếu kết quả trước và sau bản vá ở cùng số vòng/thế hệ, sau đó đo thông lượng CPU. |
| `exact` | Cận MILP cho LR108. Dùng bản IBM ILOG CPLEX có giấy phép đủ cho mô hình 50 tác vụ. |

Các benchmark so sánh dùng cùng nghiệm greedy, cùng seed và cùng giới hạn CPU. GA thuần dùng lai theo chu trình xen kẽ và tắt cải thiện cục bộ. GA memetic pilot dùng bốn lần thử lai, đột biến đổi thứ tự, xác suất đột biến thích ứng, bổ sung cá thể mới và hai bước cải thiện cục bộ. MLS tìm tiếp cho đến khi mẫu lân cận hiện tại hết bước cải thiện hoặc hết ngân sách.

In [ ]:
# 1. Tải mã và chuẩn bị môi trường
from pathlib import Path
from hashlib import sha256
import json
import os
import subprocess
import sys
import tempfile

REPO = "https://github.com/dhducpro/colab_test.git"
workspace = Path(tempfile.mkdtemp(prefix="pdpcr_hung_"))
code = workspace / "colab_test"
subprocess.run(["git", "clone", "--quiet", REPO, str(code)], check=True)
revision = subprocess.check_output(["git", "-C", str(code), "rev-parse", "HEAD"], text=True).strip()
print(f"Git revision {revision}")

env = dict(os.environ, PYTHONHASHSEED="0", PYTHONIOENCODING="utf-8")
for name in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    env[name] = "1"

probe = "import numpy, pandas, networkx, scipy, dateutil; assert int(dateutil.__version__.split('.')[0]) < 3"
if subprocess.run([sys.executable, "-c", probe], env=env, capture_output=True).returncode:
    dependencies = workspace / "dependencies"
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",
         "--target", str(dependencies), "-r", str(code / "requirements.txt")],
        env=env, check=True,
    )
    old_path = env.get("PYTHONPATH")
    env["PYTHONPATH"] = os.pathsep.join([str(dependencies), old_path] if old_path else [str(dependencies)])
subprocess.run([sys.executable, "-c", probe], env=env, check=True)
print("Mã và thư viện đã sẵn sàng.")

In [ ]:
# 2. Chọn thí nghiệm
MODE = "lr108"
CPU_SECONDS = 600   # Giới hạn mỗi seed cho lr108, families và ablation.
SEEDS = [0, 1, 2, 3, 4]
WORKERS = 5
CPLEX_SECONDS = 3600
CPLEX_EXE = ""       # Đường dẫn cplex nếu dùng CLI đã cấp phép.
COMPARE_RUNS = ""    # Đường dẫn runs.csv nếu muốn tính gap từ cận CPLEX.

In [ ]:
# 3. Chạy thí nghiệm
valid_modes = {"lr108", "pilot", "calibration", "families", "ablation",
               "claude_euclidean", "claude_grid", "performance", "exact"}
if MODE not in valid_modes:
    raise ValueError(f"MODE phải thuộc {sorted(valid_modes)}")

configuration = json.dumps([revision, MODE, float(CPU_SECONDS), SEEDS, WORKERS,
                            CPLEX_SECONDS, CPLEX_EXE, COMPARE_RUNS])
run_id = sha256(configuration.encode("utf-8")).hexdigest()[:12]
out = Path.cwd() / "pdpcr_hung_results" / f"{MODE}_{run_id}"
out.mkdir(parents=True, exist_ok=True)
print(f"Kết quả: {out}", flush=True)

if MODE.startswith("claude_"):
    script = "run2.py" if MODE == "claude_euclidean" else "run.py"
    subprocess.run([sys.executable, "-u", str(code / "vendor" / "claude" / script)],
                   cwd=out, env=env, check=True)
elif MODE == "exact":
    command = [sys.executable, "-u", str(code / "exact_reference.py"),
               str(code / "data/lr108-5-uniform.csv"), "--out", str(out), "--seconds", str(CPLEX_SECONDS)]
    if CPLEX_EXE:
        command += ["--cplex-exe", CPLEX_EXE]
    if COMPARE_RUNS:
        command += ["--runs", COMPARE_RUNS]
    subprocess.run(command, env=env, check=True)
else:
    sys.path.insert(0, str(code))
    from prepare_study import prepare

    paths = [code / "data/lr108-5-uniform.csv"]
    preset = MODE if MODE in {'pilot', 'calibration', 'ablation'} else 'quick'
    if MODE != "lr108":
        family, pilot = prepare(workspace / "instances", MODE)
        if MODE in {"families", "ablation"}:
            paths = family
        elif MODE == "pilot":
            paths = pilot
        elif MODE == "calibration":
            targets = ("lc101_n50", "lrc101_n50", "lr1_2_1_n100")
            paths = [next(p for p in pilot if tag in p.name) for tag in targets]
        else:
            targets = ("lrc101_n50", "lrc1_2_1_n100", "lrc1_4_1_n200")
            paths = [next(p for p in pilot if tag in p.name) for tag in targets]

    if MODE == "performance":
        original = workspace / "original"
        if not original.exists():
            contents = subprocess.check_output(
                ["git", "-C", str(code), "archive", "--format=zip",
                 "562c74f73c408091d190ec69a13aa757123b5eba"])
            from io import BytesIO
            from zipfile import ZipFile
            with ZipFile(BytesIO(contents)) as archive:
                archive.extractall(original)
        command = [sys.executable, "-u", str(code / "verify_speed.py"), *map(str, paths),
                   "--original", str(original), "--out", str(out / "performance.json")]
    else:
        if MODE != "lr108" and len(SEEDS) < 5:
            print("Đang chạy thử với ít hơn 5 seed.")
        command = [sys.executable, "-u", str(code / "run_hung_study.py"), "--instances", *map(str, paths),
                   "--preset", preset, "--seconds", str(CPU_SECONDS), "--workers", str(WORKERS),
                   "--seeds", *map(str, SEEDS), "--out", str(out)]
        if MODE in {"pilot", "calibration"}:
            command.append("--seconds-by-size")
    subprocess.run(command, env=env, check=True)

In [ ]:
# 4. Xem và tải kết quả
from shutil import make_archive
from IPython.display import display
import pandas as pd

for name in ("summary.csv", "tsr_throughput.csv", "wilcoxon.csv", "gaps.csv"):
    path = out / name
    if path.is_file():
        print(name)
        display(pd.read_csv(path))
for name in ("results2.jsonl", "results.jsonl"):
    path = out / name
    if path.is_file():
        display(pd.read_json(path, lines=True).drop(columns=["trace"], errors="ignore"))
if (out / "performance.json").exists():
    print((out / "performance.json").read_text(encoding="utf-8"))

archive = make_archive(str(out), "zip", root_dir=out)
try:
    from google.colab import files
except ImportError:
    print(archive)
else:
    files.download(archive)